# Analisi finale degli esperimenti e figure per la tesi

Questo notebook legge **esclusivamente** i due archivi persistenti prodotti in Colab:

- `colab_state.tar.gz`: esperimenti sul training set completo;
- `colab_subset_state.tar.gz`: esperimenti con 100, 300 e 900 immagini.

Dai TAR vengono letti in memoria soltanto i file Parquet aggregati. Checkpoint, feature e
predizioni non vengono estratti. Gli originali non vengono modificati.

**Protocollo statistico.** L'unità inferenziale è il repeat seed (`n = 10`, fold/iterazione 1).
I punti mostrano i singoli seed; i simboli grandi mostrano media e IC95% bootstrap. Per i
tempi, più asimmetrici, vengono usati mediana e IC95% bootstrap. I soli p-value visualizzati
sono quelli Holm già calcolati per Q−B2: quattro confronti primari sul full e dodici
secondari sui subset. Le altre analisi sono esplicitamente esplorative.

Gli output sono salvati in `final_analysis/figures` (PNG 300 dpi e PDF vettoriale),
`final_analysis/tables`, `captions.md` e `manifest.json`.

In [ ]:
# Configurazione Colab: questa è l'unica cella da modificare se gli archivi sono altrove.
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

DEFAULT_BUNDLE_DIR = (
    Path("/content/drive/MyDrive/tesi_qml_colab/colab_bundle")
    if IN_COLAB else Path.cwd()
)
FULL_ARCHIVE = Path(os.environ.get(
    "TESI_FULL_ARCHIVE", DEFAULT_BUNDLE_DIR / "colab_state.tar.gz"
))
SUBSET_ARCHIVE = Path(os.environ.get(
    "TESI_SUBSET_ARCHIVE", DEFAULT_BUNDLE_DIR / "colab_subset_state.tar.gz"
))
OUTPUT_DIR = Path(os.environ.get(
    "TESI_ANALYSIS_OUTPUT",
    "/content/drive/MyDrive/tesi_qml_colab/final_analysis"
    if IN_COLAB else Path.cwd() / "final_analysis",
))

FAST_MODE = os.environ.get("TESI_ANALYSIS_FAST", "0") == "1"
MAKE_ZIP = os.environ.get("TESI_ANALYSIS_ZIP", "1") != "0"
BOOTSTRAP_RESAMPLES = 300 if FAST_MODE else 10_000
EXPORT_DPI = 90 if FAST_MODE else 300

for archive_path in (FULL_ARCHIVE, SUBSET_ARCHIVE):
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Archivio non trovato: {archive_path}\n"
            "Modifica FULL_ARCHIVE/SUBSET_ARCHIVE nella prima cella."
        )

FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print({
    "full_archive": str(FULL_ARCHIVE),
    "subset_archive": str(SUBSET_ARCHIVE),
    "output": str(OUTPUT_DIR),
    "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    "dpi": EXPORT_DPI,
})

In [ ]:
# Librerie: solo stack di analisi dati e visualizzazione, oltre alla standard library.
import io
import json
import math
import shutil
import tarfile
import zlib
from datetime import datetime, timezone
from pathlib import PurePosixPath

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(context="paper", style="white", font="DejaVu Sans")
matplotlib.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": EXPORT_DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.titleweight": "semibold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.frameon": False,
    "lines.linewidth": 2.0,
    "axes.axisbelow": True,
})

TASK_LABELS = {"binary": "Binario", "three_class": "Tre classi"}
REGIME_ORDER = ("100", "300", "900", "full")
REGIME_LABELS = {"100": "100", "300": "300", "900": "900", "full": "Full"}
CONFIG_ORDER = ("B0", "B1", "B2-4", "QF-4", "Q-4", "B2-8", "QF-8", "Q-8")
MODEL_COLORS = {
    "B0": "#9A9A9A",
    "B1": "#5F6368",
    "B2": "#0072B2",
    "QF": "#E69F00",
    "Q": "#CC79A7",
}
MODEL_MARKERS = {"B0": "o", "B1": "s", "B2": "o", "QF": "^", "Q": "D"}
MODEL_LINES = {"B0": ":", "B1": "--", "B2": "-", "QF": "--", "Q": "-"}
QUBIT_MARKERS = {4: "o", 8: "s"}
EXPORTED_FIGURES = []
EXPORTED_TABLES = []

## Caricamento selettivo e controllo del protocollo

I TAR non vengono scompattati. La funzione seguente cerca il membro richiesto e passa i suoi
byte direttamente a pandas. Dal primo archivio vengono mantenute solo le 160 run `full`; dal
secondo solo le 480 run dei subset, evitando duplicazioni se il TAR dei subset contiene anche
una copia dei risultati full.

In [ ]:
def read_parquet_member(archive_path, member_suffixes):
    # Legge un singolo Parquet da un TAR senza estrarre l'archivio.
    suffixes = (member_suffixes,) if isinstance(member_suffixes, str) else tuple(member_suffixes)
    with tarfile.open(archive_path, "r:gz") as archive:
        files = [member for member in archive.getmembers() if member.isfile()]
        normalized = {
            member: PurePosixPath(member.name).as_posix().lstrip("./") for member in files
        }
        for suffix in suffixes:
            matches = [
                member for member, name in normalized.items()
                if name == suffix or name.endswith("/" + suffix)
            ]
            if len(matches) == 1:
                extracted = archive.extractfile(matches[0])
                if extracted is None:
                    raise ValueError(f"Membro illeggibile: {matches[0].name}")
                return pd.read_parquet(io.BytesIO(extracted.read()))
        available = sorted(
            name for name in normalized.values() if "results/aggregate/" in name
        )
    raise FileNotFoundError(
        f"Nessuno dei membri {suffixes} trovato in {archive_path}. "
        f"Membri aggregate disponibili: {available}"
    )


def normalize_subset_size(value):
    if pd.isna(value):
        return "full"
    text = str(value).strip().lower()
    return text[:-2] if text.endswith(".0") else text


full_runs_raw = read_parquet_member(
    FULL_ARCHIVE, "results/aggregate/runs.parquet"
)
subset_runs_raw = read_parquet_member(
    SUBSET_ARCHIVE,
    (
        "results/aggregate/runs_with_training_size.parquet",
        "results/aggregate/runs.parquet",
    ),
)
primary_hypotheses = read_parquet_member(
    FULL_ARCHIVE, "results/aggregate/primary_hypotheses.parquet"
)
secondary_hypotheses = read_parquet_member(
    SUBSET_ARCHIVE, "results/aggregate/secondary_subset_hypotheses.parquet"
)

for frame in (full_runs_raw, subset_runs_raw):
    frame["subset_size"] = frame["subset_size"].map(normalize_subset_size)

# Il TAR subset corrente contiene anche la numerosità esatta del full. Il dizionario è un
# fallback documentato per archivi precedenti che non includevano quella colonna/riga.
full_training_samples = {"binary": 1876, "three_class": 2814}
if "training_samples" in subset_runs_raw.columns:
    embedded_full = subset_runs_raw.loc[
        subset_runs_raw["subset_size"].eq("full"), ["task", "training_samples"]
    ].dropna()
    for task, group in embedded_full.groupby("task"):
        values = group["training_samples"].astype(int).unique()
        if len(values) == 1:
            full_training_samples[str(task)] = int(values[0])

full_runs = full_runs_raw.loc[full_runs_raw["subset_size"].eq("full")].copy()
full_runs["training_samples"] = full_runs["task"].map(full_training_samples).astype(int)

subset_runs = subset_runs_raw.loc[
    subset_runs_raw["subset_size"].isin(("100", "300", "900"))
].copy()
subset_runs["training_samples"] = subset_runs["subset_size"].astype(int)

runs = pd.concat([subset_runs, full_runs], ignore_index=True, sort=False)
runs["n_qubits"] = pd.to_numeric(runs["n_qubits"], errors="coerce")
runs["repeat_seed"] = pd.to_numeric(runs["repeat_seed"], errors="raise").astype(int)

def configuration_label(model, n_qubits):
    model = str(model)
    if model in {"B0", "B1"} or pd.isna(n_qubits):
        return model
    return f"{model}-{int(n_qubits)}"

runs["configuration"] = [
    configuration_label(model, qubits)
    for model, qubits in zip(runs["model"], runs["n_qubits"])
]
runs["subset_size"] = pd.Categorical(
    runs["subset_size"], categories=REGIME_ORDER, ordered=True
)
runs["configuration"] = pd.Categorical(
    runs["configuration"], categories=CONFIG_ORDER, ordered=True
)

required_columns = {
    "task", "model", "configuration", "repeat_seed", "fold", "n_qubits",
    "subset_size", "training_samples", "f1_macro", "merger_precision",
    "merger_recall", "merger_f1", "log_loss", "brier_score", "ece",
    "training_seconds", "inference_seconds",
}
missing_columns = required_columns - set(runs.columns)
if missing_columns:
    raise ValueError(f"Colonne aggregate mancanti: {sorted(missing_columns)}")

run_key = ["task", "configuration", "repeat_seed", "fold", "subset_size"]
if runs.duplicated(run_key).any():
    duplicates = runs.loc[runs.duplicated(run_key, keep=False), run_key]
    raise ValueError(f"Run duplicate dopo la fusione degli archivi:\n{duplicates.head()}")

counts = runs.groupby(
    ["task", "subset_size", "configuration"], observed=True
)["repeat_seed"].nunique()
if len(runs) != 640 or len(counts) != 64 or not counts.eq(10).all():
    raise ValueError(
        "Protocollo inatteso: sono richieste 640 run, 64 celle sperimentali e 10 seed "
        f"per cella; osservati runs={len(runs)}, celle={len(counts)}, "
        f"range_seed=({counts.min()}, {counts.max()})."
    )
if not runs["fold"].eq(1).all():
    raise ValueError("Il notebook assume il protocollo repeated holdout con fold=1.")

primary_hypotheses = primary_hypotheses.copy()
primary_hypotheses["subset_size"] = "full"
secondary_hypotheses = secondary_hypotheses.copy()
secondary_hypotheses["subset_size"] = secondary_hypotheses["subset_size"].map(
    normalize_subset_size
)
hypotheses = pd.concat(
    [
        primary_hypotheses.assign(inference_family="primario: full"),
        secondary_hypotheses.assign(inference_family="secondario: subset"),
    ],
    ignore_index=True,
    sort=False,
)

display(pd.DataFrame({
    "Archivio": ["Full", "Subset", "Combinato"],
    "Run usate": [len(full_runs), len(subset_runs), len(runs)],
    "Regimi": ["full", "100, 300, 900", "100, 300, 900, full"],
}))
print({
    "repeat_seeds": sorted(runs["repeat_seed"].unique().tolist()),
    "folds": sorted(runs["fold"].unique().tolist()),
    "full_training_samples": full_training_samples,
    "primary_hypotheses": len(primary_hypotheses),
    "secondary_hypotheses": len(secondary_hypotheses),
})

## Funzioni statistiche e sistema grafico

La palette è accessibile e coerente in tutte le figure: B0/B1 sono riferimenti grigi, B2 è
blu, QF arancione e Q magenta. Colore, marker e stile di linea sono ridondanti. I grafici a
punti non partono artificialmente da zero; tutti i pannelli confrontabili condividono invece
la stessa scala. Ogni figura viene chiusa subito dopo il salvataggio per evitare il problema
di memoria che aveva interrotto la precedente generazione batch.

In [ ]:
def stable_seed(*parts):
    payload = "|".join(map(str, parts)).encode("utf-8")
    return (20260905 + zlib.crc32(payload)) % (2**32)


def bootstrap_interval(values, *, center="mean", label=""):
    clean = np.asarray(values, dtype=float)
    clean = clean[np.isfinite(clean)]
    if clean.size == 0:
        return np.nan, np.nan, np.nan
    statistic = np.mean if center == "mean" else np.median
    estimate = float(statistic(clean))
    if clean.size == 1:
        return estimate, estimate, estimate
    rng = np.random.default_rng(stable_seed(center, label))
    samples = rng.choice(
        clean, size=(BOOTSTRAP_RESAMPLES, clean.size), replace=True
    )
    estimates = statistic(samples, axis=1)
    low, high = np.quantile(estimates, [0.025, 0.975])
    return estimate, float(low), float(high)


def summarize(frame, group_columns, value_column, *, center="mean"):
    rows = []
    grouped = frame.groupby(
        list(group_columns), dropna=False, sort=False, observed=True
    )
    for key, group in grouped:
        keys = key if isinstance(key, tuple) else (key,)
        values = pd.to_numeric(group[value_column], errors="coerce").dropna().to_numpy()
        estimate, low, high = bootstrap_interval(
            values,
            center=center,
            label="|".join(map(str, (*keys, value_column))),
        )
        rows.append({
            **dict(zip(group_columns, keys)),
            "value_column": value_column,
            "center": center,
            "estimate": estimate,
            "ci95_low": low,
            "ci95_high": high,
            "mean": float(np.mean(values)) if len(values) else np.nan,
            "median": float(np.median(values)) if len(values) else np.nan,
            "std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
            "n": int(len(values)),
        })
    return pd.DataFrame(rows)


def paired_values(frame, model_a, model_b, value_column):
    selected = frame.loc[
        frame["model"].isin((model_a, model_b)) & frame["n_qubits"].notna()
    ].copy()
    keys = ["task", "subset_size", "n_qubits", "repeat_seed", "fold"]
    pivot = selected.pivot(index=keys, columns="model", values=value_column)
    pivot = pivot.dropna(subset=[model_a, model_b]).reset_index()
    pivot["model_a"] = model_a
    pivot["model_b"] = model_b
    pivot["difference"] = pivot[model_a] - pivot[model_b]
    return pivot


def model_from_configuration(configuration):
    return str(configuration).split("-")[0]


def style_for_model(model):
    return {
        "color": MODEL_COLORS[model],
        "marker": MODEL_MARKERS[model],
        "linestyle": MODEL_LINES[model],
    }


def panel_label(axis, label):
    axis.text(
        -0.10, 1.05, label, transform=axis.transAxes,
        fontsize=12, fontweight="bold", va="bottom"
    )


def clean_axis(axis, *, grid_axis="y"):
    axis.grid(True, axis=grid_axis, color="#D9D9D9", linewidth=0.7, alpha=0.65)
    axis.grid(False, axis="x" if grid_axis == "y" else "y")
    axis.tick_params(length=3, color="#555555")


def save_table(frame, name):
    path = TABLE_DIR / f"{name}.csv"
    frame.to_csv(path, index=False, float_format="%.8g")
    EXPORTED_TABLES.append(path)
    return path


def save_figure(figure, name):
    png = FIGURE_DIR / f"{name}.png"
    pdf = FIGURE_DIR / f"{name}.pdf"
    figure.savefig(png, dpi=EXPORT_DPI, bbox_inches="tight", facecolor="white")
    figure.savefig(pdf, bbox_inches="tight", facecolor="white")
    display(figure)
    plt.close(figure)
    EXPORTED_FIGURES.append({"name": name, "png": str(png), "pdf": str(pdf)})
    print(f"Salvati: {png.name}, {pdf.name}")
    return png, pdf


def categorical_positions(regimes=REGIME_ORDER):
    return {regime: index for index, regime in enumerate(regimes)}


task_limits = {}
for task, group in runs.groupby("task", observed=True):
    low = math.floor((group["f1_macro"].min() - 0.015) * 100) / 100
    high = math.ceil((group["f1_macro"].max() + 0.012) * 100) / 100
    task_limits[task] = (max(0.0, low), min(1.0, high))

display(pd.DataFrame([
    {"task": task, "scala_macro_f1": limits}
    for task, limits in task_limits.items()
]))

# 1. Prestazioni assolute in tutti i regimi

Le quattro figure mantengono la stessa scala per ciascun task. Ogni punto piccolo è un seed;
il rombo indica la media e la linea orizzontale l'IC95% bootstrap. Le due baseline generali
B0/B1 sono intenzionalmente attenuate, mentre il confronto B2–Q resta al centro.

In [ ]:
absolute_summary = summarize(
    runs,
    ("subset_size", "task", "configuration"),
    "f1_macro",
    center="mean",
)
save_table(absolute_summary, "absolute_performance")


def plot_absolute_regime(regime, figure_name):
    figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.3), sharey=True)
    y_positions = np.arange(len(CONFIG_ORDER))[::-1]
    for column, task in enumerate(("binary", "three_class")):
        axis = axes[column]
        panel = runs.loc[
            runs["subset_size"].astype(str).eq(regime) & runs["task"].eq(task)
        ]
        summary_panel = absolute_summary.loc[
            absolute_summary["subset_size"].astype(str).eq(regime)
            & absolute_summary["task"].eq(task)
        ]
        for y, configuration in zip(y_positions, CONFIG_ORDER):
            values = panel.loc[
                panel["configuration"].astype(str).eq(configuration)
            ].sort_values("repeat_seed")["f1_macro"].to_numpy()
            model = model_from_configuration(configuration)
            color = MODEL_COLORS[model]
            jitter = np.linspace(-0.13, 0.13, len(values))
            axis.scatter(
                values, y + jitter, s=18, color=color, alpha=0.42,
                edgecolors="none", zorder=2
            )
            row = summary_panel.loc[
                summary_panel["configuration"].astype(str).eq(configuration)
            ].iloc[0]
            axis.errorbar(
                row["estimate"], y,
                xerr=[[row["estimate"] - row["ci95_low"]],
                      [row["ci95_high"] - row["estimate"]]],
                fmt="D", markersize=5.3, color=color, ecolor=color,
                capsize=3, elinewidth=1.7, markeredgecolor="white",
                markeredgewidth=0.6, zorder=4,
            )
        axis.set_xlim(task_limits[task])
        axis.set_xlabel("Macro-F1")
        axis.set_title(TASK_LABELS[task])
        axis.set_yticks(y_positions)
        axis.set_yticklabels(CONFIG_ORDER)
        clean_axis(axis, grid_axis="x")
        panel_label(axis, chr(ord("a") + column))
    figure.suptitle(
        f"Prestazioni assolute — {REGIME_LABELS[regime]} immagini di training"
        if regime != "full" else "Prestazioni assolute — training set completo",
        fontsize=14, fontweight="semibold", y=1.02,
    )
    figure.text(
        0.5, -0.015,
        "Punti: repeat seed (n=10) · rombo: media · barre: IC95% bootstrap",
        ha="center", fontsize=9, color="#444444",
    )
    figure.tight_layout()
    save_figure(figure, figure_name)


for regime, name in zip(
    REGIME_ORDER,
    ("01a_absolute_100", "01b_absolute_300", "01c_absolute_900", "01d_absolute_full"),
):
    plot_absolute_regime(regime, name)

### Perdita rispetto al full

Questa figura appaia ogni subset al full usando lo stesso seed. Un valore di `−5 pp` indica
cinque punti percentuali di Macro-F1 in meno rispetto al training completo. Le colonne
confrontano direttamente 100, 300 e 900 immagini: eventuali instabilità low-resource sono
visibili sia nella posizione dei singoli seed sia nell'ampiezza dell'intervallo.

In [ ]:
performance_wide = runs.pivot(
    index=["task", "configuration", "repeat_seed", "fold"],
    columns="subset_size",
    values="f1_macro",
).reset_index()
resource_gap_rows = []
for regime in ("100", "300", "900"):
    block = performance_wide[[
        "task", "configuration", "repeat_seed", "fold", regime, "full"
    ]].copy()
    block["subset_size"] = regime
    block["difference_pp"] = 100 * (block[regime] - block["full"])
    resource_gap_rows.append(block)
resource_gaps = pd.concat(resource_gap_rows, ignore_index=True)
resource_gap_summary = summarize(
    resource_gaps,
    ("task", "subset_size", "configuration"),
    "difference_pp",
    center="mean",
)
save_table(resource_gaps, "resource_gap_seed_values")
save_table(resource_gap_summary, "resource_gap_summary")

figure, axes = plt.subplots(2, 3, figsize=(15.5, 9.0), sharey=True)
y_positions = np.arange(len(CONFIG_ORDER))[::-1]
for row_index, task in enumerate(("binary", "three_class")):
    task_values = resource_gaps.loc[resource_gaps["task"].eq(task), "difference_pp"]
    bound = max(2.0, math.ceil(np.nanmax(np.abs(task_values)) / 2) * 2 + 1)
    for column_index, regime in enumerate(("100", "300", "900")):
        axis = axes[row_index, column_index]
        panel = resource_gaps.loc[
            resource_gaps["task"].eq(task) & resource_gaps["subset_size"].eq(regime)
        ]
        summary_panel = resource_gap_summary.loc[
            resource_gap_summary["task"].eq(task)
            & resource_gap_summary["subset_size"].eq(regime)
        ]
        for y, configuration in zip(y_positions, CONFIG_ORDER):
            values = panel.loc[
                panel["configuration"].astype(str).eq(configuration)
            ].sort_values("repeat_seed")["difference_pp"].to_numpy()
            model = model_from_configuration(configuration)
            color = MODEL_COLORS[model]
            axis.scatter(
                values, y + np.linspace(-0.12, 0.12, len(values)),
                s=15, color=color, alpha=0.38, edgecolors="none", zorder=2,
            )
            summary_row = summary_panel.loc[
                summary_panel["configuration"].astype(str).eq(configuration)
            ].iloc[0]
            axis.errorbar(
                summary_row["estimate"], y,
                xerr=[[summary_row["estimate"] - summary_row["ci95_low"]],
                      [summary_row["ci95_high"] - summary_row["estimate"]]],
                fmt="D", markersize=4.8, color=color, capsize=2.5,
                elinewidth=1.5, markeredgecolor="white", markeredgewidth=0.5,
            )
        axis.axvline(0, color="#222222", linewidth=1.0)
        axis.set_xlim(-bound, bound)
        axis.set_title(f"{TASK_LABELS[task]} · {regime} vs full")
        axis.set_xlabel("Δ Macro-F1 (punti percentuali)")
        axis.set_yticks(y_positions)
        axis.set_yticklabels(CONFIG_ORDER)
        clean_axis(axis, grid_axis="x")
        panel_label(axis, chr(ord("a") + row_index * 3 + column_index))
figure.suptitle(
    "Quanto si perde riducendo il training set?",
    fontsize=14, fontweight="semibold", y=1.01,
)
figure.text(
    0.5, -0.005,
    "Differenze appaiate sul repeat seed; valori negativi indicano prestazioni inferiori al full.",
    ha="center", fontsize=9, color="#444444",
)
figure.tight_layout()
save_figure(figure, "01e_resource_gap_vs_full")

# 2. Confronto primario appaiato Q−B2

La linea zero rappresenta la parità. Valori positivi favoriscono Q, valori negativi B2.
L'asterisco identifica esclusivamente i confronti con `p_Holm < 0,05` nelle due famiglie già
definite dal protocollo; nessun nuovo test viene effettuato nel notebook.

In [ ]:
q_minus_b2 = paired_values(runs, "Q", "B2", "f1_macro")
q_minus_b2["difference_pp"] = 100 * q_minus_b2["difference"]
q_minus_b2_summary = summarize(
    q_minus_b2,
    ("task", "subset_size", "n_qubits"),
    "difference_pp",
    center="mean",
)
hypothesis_columns = [
    "task", "subset_size", "n_qubits", "n_pairs", "mean_difference",
    "confidence_interval_95", "raw_p_value", "holm_p_value", "inference_family",
]
q_minus_b2_export = q_minus_b2.merge(
    hypotheses[hypothesis_columns],
    on=["task", "subset_size", "n_qubits"],
    how="left",
    validate="many_to_one",
)
save_table(q_minus_b2_export, "q_minus_b2_paired")
save_table(q_minus_b2_summary, "q_minus_b2_summary")
save_table(hypotheses[hypothesis_columns], "q_minus_b2_holm_tests")

positions = categorical_positions()
figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.1), sharey=True)
qubit_styles = {
    4: {"color": "#0072B2", "marker": "o", "label": "4 qubit"},
    8: {"color": "#7A5195", "marker": "s", "label": "8 qubit"},
}
for panel_index, task in enumerate(("binary", "three_class")):
    axis = axes[panel_index]
    for q_index, qubits in enumerate((4, 8)):
        style = qubit_styles[qubits]
        offset = -0.11 if qubits == 4 else 0.11
        series_x, series_y = [], []
        for regime in REGIME_ORDER:
            x = positions[regime] + offset
            values = q_minus_b2.loc[
                q_minus_b2["task"].eq(task)
                & q_minus_b2["subset_size"].astype(str).eq(regime)
                & q_minus_b2["n_qubits"].eq(qubits),
                "difference_pp",
            ].sort_index().to_numpy()
            axis.scatter(
                x + np.linspace(-0.035, 0.035, len(values)), values,
                s=17, color=style["color"], alpha=0.32, edgecolors="none",
            )
            summary_row = q_minus_b2_summary.loc[
                q_minus_b2_summary["task"].eq(task)
                & q_minus_b2_summary["subset_size"].astype(str).eq(regime)
                & q_minus_b2_summary["n_qubits"].eq(qubits)
            ].iloc[0]
            axis.errorbar(
                x, summary_row["estimate"],
                yerr=[[summary_row["estimate"] - summary_row["ci95_low"]],
                      [summary_row["ci95_high"] - summary_row["estimate"]]],
                fmt=style["marker"], color=style["color"], capsize=3,
                markersize=5.5, markeredgecolor="white", markeredgewidth=0.6,
            )
            series_x.append(x)
            series_y.append(summary_row["estimate"])
            h_row = hypotheses.loc[
                hypotheses["task"].eq(task)
                & hypotheses["subset_size"].astype(str).eq(regime)
                & hypotheses["n_qubits"].eq(qubits)
            ]
            if len(h_row) == 1 and float(h_row.iloc[0]["holm_p_value"]) < 0.05:
                y_star = summary_row["ci95_high"] + 0.35
                axis.text(x, y_star, "*", ha="center", va="bottom",
                          color=style["color"], fontsize=14, fontweight="bold")
        axis.plot(series_x, series_y, color=style["color"], alpha=0.7,
                  linewidth=1.5, linestyle="-", label=style["label"])
    axis.axhline(0, color="#222222", linewidth=1.1)
    axis.set_xticks(range(len(REGIME_ORDER)), [REGIME_LABELS[x] for x in REGIME_ORDER])
    axis.set_xlabel("Immagini di training")
    axis.set_title(TASK_LABELS[task])
    clean_axis(axis, grid_axis="y")
    panel_label(axis, chr(ord("a") + panel_index))
axes[0].set_ylabel("Q − B2 in Macro-F1 (punti percentuali)")
axes[1].legend(loc="lower right")
figure.suptitle(
    "Il vantaggio quantistico emerge rispetto al controllo classico appaiato?",
    fontsize=14, fontweight="semibold", y=1.02,
)
figure.text(
    0.5, -0.015,
    "Punti: seed appaiati · simbolo: media · IC95% bootstrap · * p Holm < 0,05 già calcolato",
    ha="center", fontsize=9, color="#444444",
)
figure.tight_layout()
save_figure(figure, "02_q_minus_b2_paired")

# 3. Curve di apprendimento rispetto alla numerosità del training

Il vero numero di immagini è usato sull'asse x: 100, 300, 900 e full (`1.876` nel binario,
`2.814` nel task a tre classi). La scala logaritmica rispetta le distanze moltiplicative tra i
regimi. I punti tenui sono i seed, le linee uniscono le medie e le bande mostrano gli IC95%.
B2 e Q sono il confronto principale; QF resta una diagnosi secondaria del circuito.

In [ ]:
learning_runs = runs.loc[runs["model"].isin(("B2", "QF", "Q"))].copy()
learning_summary = summarize(
    learning_runs,
    ("task", "n_qubits", "model", "subset_size", "training_samples"),
    "f1_macro",
    center="mean",
)
save_table(learning_summary, "learning_curves")

figure, axes = plt.subplots(2, 2, figsize=(13.4, 9.2), sharex=False, sharey=False)
for row_index, task in enumerate(("binary", "three_class")):
    for column_index, qubits in enumerate((4, 8)):
        axis = axes[row_index, column_index]
        panel_runs = learning_runs.loc[
            learning_runs["task"].eq(task) & learning_runs["n_qubits"].eq(qubits)
        ]
        panel_summary = learning_summary.loc[
            learning_summary["task"].eq(task)
            & learning_summary["n_qubits"].eq(qubits)
        ]
        for model in ("B2", "QF", "Q"):
            style = style_for_model(model)
            summary_model = panel_summary.loc[
                panel_summary["model"].eq(model)
            ].sort_values("training_samples")
            x = summary_model["training_samples"].to_numpy(dtype=float)
            y = summary_model["estimate"].to_numpy(dtype=float)
            low = summary_model["ci95_low"].to_numpy(dtype=float)
            high = summary_model["ci95_high"].to_numpy(dtype=float)
            for sample_size in x:
                raw = panel_runs.loc[
                    panel_runs["model"].eq(model)
                    & panel_runs["training_samples"].eq(sample_size),
                    "f1_macro",
                ].to_numpy()
                axis.scatter(
                    np.full(len(raw), sample_size), raw, s=13,
                    color=style["color"], alpha=0.20, edgecolors="none",
                )
            axis.fill_between(x, low, high, color=style["color"], alpha=0.12)
            axis.plot(
                x, y, label=model, color=style["color"], marker=style["marker"],
                linestyle=style["linestyle"], markersize=5.5,
            )
        full_n = full_training_samples[task]
        axis.set_xscale("log")
        axis.set_xticks([100, 300, 900, full_n])
        axis.set_xticklabels(["100", "300", "900", f"Full\n({full_n:,})".replace(",", ".")])
        axis.xaxis.set_minor_formatter(NullFormatter())
        axis.set_ylim(task_limits[task])
        axis.set_xlabel("Immagini di training (scala log)")
        axis.set_ylabel("Macro-F1" if column_index == 0 else "")
        axis.set_title(f"{TASK_LABELS[task]} · {qubits} qubit")
        clean_axis(axis, grid_axis="y")
        panel_label(axis, chr(ord("a") + row_index * 2 + column_index))
handles = [
    Line2D([0], [0], label=model, color=MODEL_COLORS[model],
           marker=MODEL_MARKERS[model], linestyle=MODEL_LINES[model])
    for model in ("B2", "QF", "Q")
]
figure.legend(handles=handles, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.01))
figure.suptitle(
    "Curve di apprendimento: il confronto B2–Q cambia nel regime low-resource?",
    fontsize=14, fontweight="semibold", y=1.05,
)
figure.text(
    0.5, -0.005,
    "Punti: repeat seed (n=10) · linea: media · banda: IC95% bootstrap",
    ha="center", fontsize=9, color="#444444",
)
figure.tight_layout()
save_figure(figure, "03_learning_curves")

# 4. Dal binario alla terza classe: dove divergono classico e quantistico?

Questa sezione è esplorativa e approfondisce quattro domande distinte:

1. quanto perde ogni architettura passando dal binario alle tre classi;
2. se il divario Q−B2 peggiora specificamente nel task più complesso;
3. se la classe `merger` spiega parte del comportamento;
4. se l'addestramento dei parametri quantistici (Q rispetto a QF) cambia il risultato.

Gli intervalli quantificano gli effetti; non vengono aggiunti nuovi p-value.

In [ ]:
task_wide = learning_runs.pivot(
    index=["subset_size", "model", "n_qubits", "repeat_seed", "fold"],
    columns="task",
    values="f1_macro",
).dropna(subset=["binary", "three_class"]).reset_index()
task_wide["difference_pp"] = 100 * (task_wide["three_class"] - task_wide["binary"])
task_transition_summary = summarize(
    task_wide,
    ("n_qubits", "model", "subset_size"),
    "difference_pp",
    center="mean",
)
save_table(task_wide, "binary_to_three_class")
save_table(task_transition_summary, "binary_to_three_class_summary")

positions = categorical_positions()
figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.2), sharey=True)
for panel_index, qubits in enumerate((4, 8)):
    axis = axes[panel_index]
    for model_index, model in enumerate(("B2", "QF", "Q")):
        style = style_for_model(model)
        offset = (-0.16, 0.0, 0.16)[model_index]
        xs, ys = [], []
        for regime in REGIME_ORDER:
            x = positions[regime] + offset
            raw = task_wide.loc[
                task_wide["n_qubits"].eq(qubits)
                & task_wide["model"].eq(model)
                & task_wide["subset_size"].astype(str).eq(regime),
                "difference_pp",
            ].to_numpy()
            axis.scatter(
                x + np.linspace(-0.025, 0.025, len(raw)), raw,
                s=14, color=style["color"], alpha=0.25, edgecolors="none",
            )
            row = task_transition_summary.loc[
                task_transition_summary["n_qubits"].eq(qubits)
                & task_transition_summary["model"].eq(model)
                & task_transition_summary["subset_size"].astype(str).eq(regime)
            ].iloc[0]
            axis.errorbar(
                x, row["estimate"],
                yerr=[[row["estimate"] - row["ci95_low"]],
                      [row["ci95_high"] - row["estimate"]]],
                fmt=style["marker"], color=style["color"], markersize=5,
                capsize=2.5, markeredgecolor="white", markeredgewidth=0.5,
            )
            xs.append(x)
            ys.append(row["estimate"])
        axis.plot(xs, ys, color=style["color"], linestyle=style["linestyle"],
                  linewidth=1.6, label=model)
    axis.axhline(0, color="#222222", linewidth=1.0)
    axis.set_xticks(range(4), [REGIME_LABELS[x] for x in REGIME_ORDER])
    axis.set_xlabel("Immagini di training")
    axis.set_title(f"{qubits} qubit")
    clean_axis(axis, grid_axis="y")
    panel_label(axis, chr(ord("a") + panel_index))
axes[0].set_ylabel("Tre classi − binario (punti percentuali di Macro-F1)")
axes[1].legend(loc="lower right")
figure.suptitle(
    "Penalità osservata introducendo la classe merger",
    fontsize=14, fontweight="semibold", y=1.02,
)
figure.text(
    0.5, -0.015,
    "Appaiamento per repeat seed; valori più negativi indicano una perdita maggiore nel task a tre classi.",
    ha="center", fontsize=9, color="#444444",
)
figure.tight_layout()
save_figure(figure, "04a_binary_to_three_class")

### Cambiamento relativo del divario quantistico

La differenza-in-differenze è
`(Q−B2) tre classi − (Q−B2) binario`. Un valore negativo significa che Q perde terreno
rispetto al controllo classico quando viene introdotta la classe merger.

In [ ]:
gap_by_task = q_minus_b2.pivot(
    index=["subset_size", "n_qubits", "repeat_seed", "fold"],
    columns="task",
    values="difference",
).dropna(subset=["binary", "three_class"]).reset_index()
gap_by_task["difference_in_difference_pp"] = 100 * (
    gap_by_task["three_class"] - gap_by_task["binary"]
)
relative_penalty_summary = summarize(
    gap_by_task,
    ("n_qubits", "subset_size"),
    "difference_in_difference_pp",
    center="mean",
)
save_table(gap_by_task, "relative_quantum_penalty")
save_table(relative_penalty_summary, "relative_quantum_penalty_summary")

figure, axis = plt.subplots(figsize=(8.6, 5.2))
for qubits, color, marker, offset in (
    (4, "#0072B2", "o", -0.09),
    (8, "#7A5195", "s", 0.09),
):
    xs, ys = [], []
    for regime in REGIME_ORDER:
        x = positions[regime] + offset
        raw = gap_by_task.loc[
            gap_by_task["n_qubits"].eq(qubits)
            & gap_by_task["subset_size"].astype(str).eq(regime),
            "difference_in_difference_pp",
        ].to_numpy()
        axis.scatter(
            x + np.linspace(-0.03, 0.03, len(raw)), raw,
            s=18, color=color, alpha=0.30, edgecolors="none",
        )
        row = relative_penalty_summary.loc[
            relative_penalty_summary["n_qubits"].eq(qubits)
            & relative_penalty_summary["subset_size"].astype(str).eq(regime)
        ].iloc[0]
        axis.errorbar(
            x, row["estimate"],
            yerr=[[row["estimate"] - row["ci95_low"]],
                  [row["ci95_high"] - row["estimate"]]],
            fmt=marker, color=color, capsize=3, markersize=6,
            markeredgecolor="white", markeredgewidth=0.6,
        )
        xs.append(x)
        ys.append(row["estimate"])
    axis.plot(xs, ys, color=color, linewidth=1.6, label=f"{qubits} qubit")
axis.axhline(0, color="#222222", linewidth=1.1)
axis.set_xticks(range(4), [REGIME_LABELS[x] for x in REGIME_ORDER])
axis.set_xlabel("Immagini di training")
axis.set_ylabel("Δ del divario Q−B2 (punti percentuali)")
axis.set_title("Q perde terreno rispetto a B2 passando al task a tre classi?")
axis.legend()
clean_axis(axis, grid_axis="y")
figure.text(
    0.5, -0.015,
    "Valori negativi: il divario Q−B2 è più sfavorevole nel task a tre classi; analisi esplorativa.",
    ha="center", fontsize=9, color="#444444",
)
figure.tight_layout()
save_figure(figure, "04b_relative_quantum_penalty")

### Classe merger: precision, recall e F1

I sei pannelli separano numero di qubit e metrica per non sovrapporre scale o significati.
La lettura congiunta di precision e recall permette di capire se un calo di F1 dipende da
merger non riconosciute oppure da falsi positivi.

In [ ]:
merger_metrics = ("merger_precision", "merger_recall", "merger_f1")
merger_runs = runs.loc[
    runs["task"].eq("three_class") & runs["model"].isin(("B2", "QF", "Q"))
].copy()
merger_summaries = []
for metric in merger_metrics:
    block = summarize(
        merger_runs,
        ("n_qubits", "model", "subset_size", "training_samples"),
        metric,
        center="mean",
    )
    merger_summaries.append(block)
merger_summary = pd.concat(merger_summaries, ignore_index=True)
save_table(merger_summary, "merger_detailed")

metric_labels = {
    "merger_precision": "Precision merger",
    "merger_recall": "Recall merger",
    "merger_f1": "F1 merger",
}
merger_low = max(0, math.floor((merger_runs[list(merger_metrics)].min().min() - 0.03) * 20) / 20)
merger_high = min(1, math.ceil((merger_runs[list(merger_metrics)].max().max() + 0.03) * 20) / 20)
figure, axes = plt.subplots(2, 3, figsize=(15.4, 8.8), sharex=False, sharey=True)
for row_index, qubits in enumerate((4, 8)):
    for column_index, metric in enumerate(merger_metrics):
        axis = axes[row_index, column_index]
        panel = merger_summary.loc[
            merger_summary["n_qubits"].eq(qubits)
            & merger_summary["value_column"].eq(metric)
        ]
        raw_panel = merger_runs.loc[merger_runs["n_qubits"].eq(qubits)]
        for model in ("B2", "QF", "Q"):
            style = style_for_model(model)
            model_summary = panel.loc[panel["model"].eq(model)].sort_values("training_samples")
            x = model_summary["training_samples"].to_numpy(dtype=float)
            y = model_summary["estimate"].to_numpy(dtype=float)
            low = model_summary["ci95_low"].to_numpy(dtype=float)
            high = model_summary["ci95_high"].to_numpy(dtype=float)
            for sample_size in x:
                raw = raw_panel.loc[
                    raw_panel["model"].eq(model)
                    & raw_panel["training_samples"].eq(sample_size), metric
                ].dropna().to_numpy()
                axis.scatter(
                    np.full(len(raw), sample_size), raw, s=10,
                    color=style["color"], alpha=0.18, edgecolors="none",
                )
            axis.fill_between(x, low, high, color=style["color"], alpha=0.10)
            axis.plot(
                x, y, color=style["color"], marker=style["marker"],
                linestyle=style["linestyle"], markersize=5, label=model,
            )
        full_n = full_training_samples["three_class"]
        axis.set_xscale("log")
        axis.set_xticks([100, 300, 900, full_n])
        axis.set_xticklabels(["100", "300", "900", "Full"])
        axis.xaxis.set_minor_formatter(NullFormatter())
        axis.set_ylim(merger_low, merger_high)
        axis.set_title(f"{qubits} qubit · {metric_labels[metric]}")
        axis.set_xlabel("Immagini di training (scala log)")
        if column_index == 0:
            axis.set_ylabel("Score")
        clean_axis(axis, grid_axis="y")
        panel_label(axis, chr(ord("a") + row_index * 3 + column_index))
handles = [
    Line2D([0], [0], label=model, color=MODEL_COLORS[model],
           marker=MODEL_MARKERS[model], linestyle=MODEL_LINES[model])
    for model in ("B2", "QF", "Q")
]
figure.legend(handles=handles, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.01))
figure.suptitle(
    "Prestazioni sulla classe merger nei diversi regimi di dati",
    fontsize=14, fontweight="semibold", y=1.05,
)
figure.text(
    0.5, -0.005,
    "Punti: repeat seed (n=10) · linea: media · banda: IC95% bootstrap · analisi esplorativa",
    ha="center", fontsize=9, color="#444444",
)
figure.tight_layout()
save_figure(figure, "04c_merger_detailed")

### Q rispetto a QF

Q e QF condividono il circuito; in Q i parametri quantistici sono addestrabili, in QF sono
congelati. Il grafico misura quindi l'effetto empirico dell'ottimizzazione del circuito,
senza trasformarlo in un nuovo test confermatorio.

In [ ]:
q_minus_qf = paired_values(runs, "Q", "QF", "f1_macro")
q_minus_qf["difference_pp"] = 100 * q_minus_qf["difference"]
q_minus_qf_summary = summarize(
    q_minus_qf,
    ("task", "subset_size", "n_qubits"),
    "difference_pp",
    center="mean",
)
save_table(q_minus_qf, "q_minus_qf")
save_table(q_minus_qf_summary, "q_minus_qf_summary")

figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True)
for panel_index, task in enumerate(("binary", "three_class")):
    axis = axes[panel_index]
    for qubits, color, marker, offset in (
        (4, "#0072B2", "o", -0.09),
        (8, "#7A5195", "s", 0.09),
    ):
        xs, ys = [], []
        for regime in REGIME_ORDER:
            x = positions[regime] + offset
            raw = q_minus_qf.loc[
                q_minus_qf["task"].eq(task)
                & q_minus_qf["n_qubits"].eq(qubits)
                & q_minus_qf["subset_size"].astype(str).eq(regime),
                "difference_pp",
            ].to_numpy()
            axis.scatter(x + np.linspace(-0.03, 0.03, len(raw)), raw,
                         s=16, color=color, alpha=0.28, edgecolors="none")
            summary_row = q_minus_qf_summary.loc[
                q_minus_qf_summary["task"].eq(task)
                & q_minus_qf_summary["n_qubits"].eq(qubits)
                & q_minus_qf_summary["subset_size"].astype(str).eq(regime)
            ].iloc[0]
            axis.errorbar(
                x, summary_row["estimate"],
                yerr=[[summary_row["estimate"] - summary_row["ci95_low"]],
                      [summary_row["ci95_high"] - summary_row["estimate"]]],
                fmt=marker, color=color, capsize=3, markersize=5.5,
                markeredgecolor="white", markeredgewidth=0.6,
            )
            xs.append(x)
            ys.append(summary_row["estimate"])
        axis.plot(xs, ys, color=color, linewidth=1.5, label=f"{qubits} qubit")
    axis.axhline(0, color="#222222", linewidth=1.0)
    axis.set_xticks(range(4), [REGIME_LABELS[x] for x in REGIME_ORDER])
    axis.set_xlabel("Immagini di training")
    axis.set_title(TASK_LABELS[task])
    clean_axis(axis, grid_axis="y")
    panel_label(axis, chr(ord("a") + panel_index))
axes[0].set_ylabel("Q − QF in Macro-F1 (punti percentuali)")
axes[1].legend()
figure.suptitle(
    "Quanto contribuisce l'addestramento dei parametri quantistici?",
    fontsize=14, fontweight="semibold", y=1.02,
)
figure.text(
    0.5, -0.015,
    "Valori positivi favoriscono Q; effetti e IC95% bootstrap, senza nuovi p-value.",
    ha="center", fontsize=9, color="#444444",
)
figure.tight_layout()
save_figure(figure, "04d_q_minus_qf")

### Calibrazione e qualità probabilistica

Log-loss, Brier score ed ECE sono metriche di errore: valori inferiori sono migliori. Nei
pannelli seguenti `Q−B2 > 0` indica quindi un peggioramento del modello quantistico. La
separazione per task evita di confondere la maggiore difficoltà assoluta delle tre classi con
la differenza tra architetture.

In [ ]:
calibration_metrics = ("log_loss", "brier_score", "ece")
calibration_blocks = []
for metric in calibration_metrics:
    block = paired_values(runs, "Q", "B2", metric)
    block["metric"] = metric
    calibration_blocks.append(block)
calibration_differences = pd.concat(calibration_blocks, ignore_index=True)
calibration_summary = summarize(
    calibration_differences,
    ("task", "n_qubits", "subset_size", "metric"),
    "difference",
    center="mean",
)
save_table(calibration_differences, "calibration_q_minus_b2")
save_table(calibration_summary, "calibration_q_minus_b2_summary")

calibration_labels = {
    "log_loss": "Δ log-loss",
    "brier_score": "Δ Brier score",
    "ece": "Δ ECE",
}
figure, axes = plt.subplots(2, 3, figsize=(15.2, 8.4), sharex=True, sharey=False)
for row_index, task in enumerate(("binary", "three_class")):
    for column_index, metric in enumerate(calibration_metrics):
        axis = axes[row_index, column_index]
        for qubits, color, marker, offset in (
            (4, "#0072B2", "o", -0.07),
            (8, "#7A5195", "s", 0.07),
        ):
            xs, ys = [], []
            for regime in REGIME_ORDER:
                x = positions[regime] + offset
                row = calibration_summary.loc[
                    calibration_summary["task"].eq(task)
                    & calibration_summary["metric"].eq(metric)
                    & calibration_summary["n_qubits"].eq(qubits)
                    & calibration_summary["subset_size"].astype(str).eq(regime)
                ].iloc[0]
                axis.errorbar(
                    x, row["estimate"],
                    yerr=[[row["estimate"] - row["ci95_low"]],
                          [row["ci95_high"] - row["estimate"]]],
                    fmt=marker, color=color, capsize=2.5, markersize=5.2,
                    markeredgecolor="white", markeredgewidth=0.5,
                )
                xs.append(x)
                ys.append(row["estimate"])
            axis.plot(xs, ys, color=color, linewidth=1.5, label=f"{qubits} qubit")
        axis.axhline(0, color="#222222", linewidth=1.0)
        axis.set_xticks(range(4), [REGIME_LABELS[x] for x in REGIME_ORDER])
        axis.set_title(f"{TASK_LABELS[task]} · {calibration_labels[metric]}")
        if row_index == 1:
            axis.set_xlabel("Immagini di training")
        if column_index == 0:
            axis.set_ylabel("Q − B2")
        clean_axis(axis, grid_axis="y")
        panel_label(axis, chr(ord("a") + row_index * 3 + column_index))
axes[0, 2].legend()
figure.suptitle(
    "Il modello quantistico cambia calibrazione e qualità probabilistica?",
    fontsize=14, fontweight="semibold", y=1.03,
)
figure.text(
    0.5, -0.005,
    "Tutte le metriche sono errori: valori positivi favoriscono B2; media e IC95% bootstrap, analisi esplorativa.",
    ha="center", fontsize=9, color="#444444",
)
figure.tight_layout()
save_figure(figure, "04e_calibration")

# 5. Costo computazionale

I rapporti sono calcolati a parità di task, subset, qubit e seed. `1×` indica lo stesso costo
di B2. Poiché i tempi sono positivi e asimmetrici, il simbolo grande è la **mediana** e
l'intervallo è un IC95% bootstrap della mediana; i punti mostrano tutti i seed. L'asse y è
logaritmico. Il tempo totale di training include l'effetto dell'early stopping e va quindi
interpretato come costo empirico del protocollo, non come sola complessità dell'architettura.

In [ ]:
cost_blocks = []
cost_keys = ["task", "subset_size", "n_qubits", "repeat_seed", "fold"]
for metric in ("training_seconds", "inference_seconds"):
    for numerator in ("QF", "Q"):
        selected = runs.loc[
            runs["model"].isin(("B2", numerator)) & runs["n_qubits"].notna()
        ]
        pivot = selected.pivot(index=cost_keys, columns="model", values=metric)
        pivot = pivot.dropna(subset=["B2", numerator]).reset_index()
        pivot["numerator_model"] = numerator
        pivot["cost_metric"] = metric
        pivot["ratio_to_b2"] = pivot[numerator] / pivot["B2"]
        cost_blocks.append(pivot)
cost_ratios = pd.concat(cost_blocks, ignore_index=True)
cost_summary = summarize(
    cost_ratios,
    ("task", "cost_metric", "numerator_model", "n_qubits", "subset_size"),
    "ratio_to_b2",
    center="median",
)
save_table(cost_ratios, "cost_ratios")
save_table(cost_summary, "cost_ratios_summary")

cost_metric_labels = {
    "training_seconds": "Tempo totale di training",
    "inference_seconds": "Tempo di inferenza",
}
figure, axes = plt.subplots(2, 2, figsize=(13.8, 9.0), sharex=True, sharey=True)
for row_index, task in enumerate(("binary", "three_class")):
    for column_index, metric in enumerate(("training_seconds", "inference_seconds")):
        axis = axes[row_index, column_index]
        for model_index, numerator in enumerate(("QF", "Q")):
            for q_index, qubits in enumerate((4, 8)):
                color = MODEL_COLORS[numerator]
                marker = QUBIT_MARKERS[qubits]
                linestyle = "-" if qubits == 4 else "--"
                offset = (-0.18, -0.06, 0.06, 0.18)[model_index * 2 + q_index]
                xs, ys = [], []
                for regime in REGIME_ORDER:
                    x = positions[regime] + offset
                    raw = cost_ratios.loc[
                        cost_ratios["task"].eq(task)
                        & cost_ratios["cost_metric"].eq(metric)
                        & cost_ratios["numerator_model"].eq(numerator)
                        & cost_ratios["n_qubits"].eq(qubits)
                        & cost_ratios["subset_size"].astype(str).eq(regime),
                        "ratio_to_b2",
                    ].to_numpy()
                    axis.scatter(
                        x + np.linspace(-0.018, 0.018, len(raw)), raw,
                        s=11, color=color, alpha=0.20, edgecolors="none",
                    )
                    summary_row = cost_summary.loc[
                        cost_summary["task"].eq(task)
                        & cost_summary["cost_metric"].eq(metric)
                        & cost_summary["numerator_model"].eq(numerator)
                        & cost_summary["n_qubits"].eq(qubits)
                        & cost_summary["subset_size"].astype(str).eq(regime)
                    ].iloc[0]
                    axis.errorbar(
                        x, summary_row["estimate"],
                        yerr=[[summary_row["estimate"] - summary_row["ci95_low"]],
                              [summary_row["ci95_high"] - summary_row["estimate"]]],
                        fmt=marker, color=color, capsize=2.5, markersize=5.2,
                        markeredgecolor="white", markeredgewidth=0.5,
                    )
                    xs.append(x)
                    ys.append(summary_row["estimate"])
                axis.plot(
                    xs, ys, color=color, linestyle=linestyle, linewidth=1.5,
                    label=f"{numerator}/B2 · {qubits}q",
                )
        axis.axhline(1, color="#222222", linewidth=1.0)
        axis.set_yscale("log")
        axis.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}×"))
        axis.set_xticks(range(4), [REGIME_LABELS[x] for x in REGIME_ORDER])
        axis.set_title(f"{TASK_LABELS[task]} · {cost_metric_labels[metric]}")
        if row_index == 1:
            axis.set_xlabel("Immagini di training")
        if column_index == 0:
            axis.set_ylabel("Rapporto rispetto a B2 (scala log)")
        clean_axis(axis, grid_axis="y")
        panel_label(axis, chr(ord("a") + row_index * 2 + column_index))
axes[0, 1].legend(loc="upper left", ncol=2, fontsize=8)
figure.suptitle(
    "Costo empirico del circuito simulato rispetto al controllo classico",
    fontsize=14, fontweight="semibold", y=1.03,
)
figure.text(
    0.5, -0.005,
    "Punti: seed appaiati · simbolo: mediana · IC95% bootstrap della mediana · asse y logaritmico",
    ha="center", fontsize=9, color="#444444",
)
figure.tight_layout()
save_figure(figure, "05_cost_ratios")

# Appendice A. Sensibilità a 4 o 8 qubit

Questo confronto è tenuto separato dalla narrativa principale, così può essere usato nel
capitolo dedicato all'architettura quantistica. `8−4 > 0` favorisce la configurazione a otto
qubit. Sono mostrati effetti e intervalli, senza nuovi p-value.

In [ ]:
qubit_runs = runs.loc[runs["model"].isin(("B2", "QF", "Q"))].copy()
qubit_wide = qubit_runs.pivot(
    index=["task", "subset_size", "model", "repeat_seed", "fold"],
    columns="n_qubits",
    values="f1_macro",
).dropna(subset=[4.0, 8.0]).reset_index()
qubit_wide["difference_pp"] = 100 * (qubit_wide[8.0] - qubit_wide[4.0])
qubit_summary = summarize(
    qubit_wide,
    ("task", "model", "subset_size"),
    "difference_pp",
    center="mean",
)
save_table(qubit_wide, "qubit_8_minus_4")
save_table(qubit_summary, "qubit_8_minus_4_summary")

figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True)
for panel_index, task in enumerate(("binary", "three_class")):
    axis = axes[panel_index]
    for model_index, model in enumerate(("B2", "QF", "Q")):
        style = style_for_model(model)
        offset = (-0.14, 0.0, 0.14)[model_index]
        xs, ys = [], []
        for regime in REGIME_ORDER:
            x = positions[regime] + offset
            raw = qubit_wide.loc[
                qubit_wide["task"].eq(task)
                & qubit_wide["model"].eq(model)
                & qubit_wide["subset_size"].astype(str).eq(regime),
                "difference_pp",
            ].to_numpy()
            axis.scatter(x + np.linspace(-0.025, 0.025, len(raw)), raw,
                         s=14, color=style["color"], alpha=0.25, edgecolors="none")
            row = qubit_summary.loc[
                qubit_summary["task"].eq(task)
                & qubit_summary["model"].eq(model)
                & qubit_summary["subset_size"].astype(str).eq(regime)
            ].iloc[0]
            axis.errorbar(
                x, row["estimate"],
                yerr=[[row["estimate"] - row["ci95_low"]],
                      [row["ci95_high"] - row["estimate"]]],
                fmt=style["marker"], color=style["color"], capsize=2.5,
                markersize=5, markeredgecolor="white", markeredgewidth=0.5,
            )
            xs.append(x)
            ys.append(row["estimate"])
        axis.plot(xs, ys, color=style["color"], linestyle=style["linestyle"],
                  linewidth=1.5, label=model)
    axis.axhline(0, color="#222222", linewidth=1.0)
    axis.set_xticks(range(4), [REGIME_LABELS[x] for x in REGIME_ORDER])
    axis.set_xlabel("Immagini di training")
    axis.set_title(TASK_LABELS[task])
    clean_axis(axis, grid_axis="y")
    panel_label(axis, chr(ord("a") + panel_index))
axes[0].set_ylabel("8 qubit − 4 qubit (punti percentuali di Macro-F1)")
axes[1].legend()
figure.suptitle(
    "Sensibilità della Macro-F1 al numero di qubit",
    fontsize=14, fontweight="semibold", y=1.02,
)
figure.text(
    0.5, -0.015,
    "Differenze appaiate sul repeat seed; valori positivi favoriscono 8 qubit; analisi esplorativa.",
    ha="center", fontsize=9, color="#444444",
)
figure.tight_layout()
save_figure(figure, "appendix_a_qubit_sensitivity")

# Esportazione delle didascalie e riepilogo

Le didascalie seguenti sono volutamente autosufficienti e possono essere adattate nel testo
della tesi. Il manifest registra input, versioni e file prodotti.

In [ ]:
captions = {
    "01a–01d | Prestazioni assolute": (
        "Macro-F1 delle otto configurazioni nei regimi 100, 300, 900 e full, separata "
        "per task. I punti piccoli rappresentano i dieci repeat seed; il rombo è la media "
        "e le barre sono IC95% bootstrap (10.000 ricampionamenti). L'unità statistica è il "
        "repeat seed; tutte le run appartengono al fold/iterazione 1. B0 e B1 sono riferimenti "
        "contestuali, mentre B2 e Q costituiscono il confronto centrale."
    ),
    "01e | Perdita rispetto al full": (
        "Differenza appaiata tra ciascun subset e il full a parità di configurazione e seed. "
        "Valori negativi indicano una perdita di Macro-F1; punti, media e IC95% rendono "
        "visibili sia l'effetto medio sia eventuali seed instabili nei regimi low-resource."
    ),
    "02 | Q−B2": (
        "Differenza appaiata di Macro-F1 tra modello quantistico addestrabile Q e controllo "
        "classico a capacità appaiata B2. Valori positivi favoriscono Q. Gli asterischi "
        "indicano p Holm < 0,05 usando esclusivamente le quattro ipotesi primarie sul full e "
        "le dodici secondarie sui subset già predefinite; gli IC95% sono bootstrap."
    ),
    "03 | Curve di apprendimento": (
        "Macro-F1 rispetto al numero effettivo di immagini di training per B2, QF e Q. "
        "L'asse x è logaritmico; i punti sono i repeat seed, le linee sono le medie e le "
        "bande gli IC95% bootstrap. Le scale y sono condivise tra i pannelli dello stesso task."
    ),
    "04a | Passaggio binario–tre classi": (
        "Variazione di Macro-F1 passando dal task binario al task a tre classi, appaiata per "
        "repeat seed. Valori più negativi indicano una penalità maggiore associata "
        "all'introduzione della classe merger. Analisi esplorativa basata su effetti e IC95%."
    ),
    "04b | Penalità quantistica relativa": (
        "Differenza-in-differenze [(Q−B2) tre classi − (Q−B2) binario]. Valori negativi "
        "indicano che Q perde terreno rispetto a B2 nel task più complesso. Non sono stati "
        "calcolati nuovi p-value."
    ),
    "04c | Classe merger": (
        "Precision, recall e F1 della classe merger per B2, QF e Q nei quattro regimi di "
        "training. Punti, medie e IC95% bootstrap sono mostrati separatamente per 4 e 8 qubit."
    ),
    "04d | Q−QF": (
        "Differenza appaiata Q−QF in Macro-F1. Valori positivi indicano un beneficio "
        "dell'addestramento dei parametri quantistici rispetto allo stesso circuito congelato. "
        "Analisi esplorativa con effetti e IC95%, senza nuovi test."
    ),
    "04e | Calibrazione": (
        "Differenze appaiate Q−B2 per log-loss, Brier score ed expected calibration error. "
        "Poiché tutte sono metriche di errore, valori positivi favoriscono B2. Sono riportate "
        "medie e IC95% bootstrap; l'analisi è esplorativa."
    ),
    "05 | Costo computazionale": (
        "Rapporto dei tempi QF/B2 e Q/B2 a parità di task, regime, qubit e repeat seed. "
        "La linea 1× indica parità di costo; i simboli sono mediane, gli intervalli IC95% "
        "bootstrap della mediana e l'asse y è logaritmico. Il training include early stopping."
    ),
    "Appendice A | 8−4 qubit": (
        "Differenza appaiata di Macro-F1 tra configurazioni a 8 e 4 qubit. Valori positivi "
        "favoriscono 8 qubit. Il confronto è esplorativo e separato dalla narrativa primaria."
    ),
}

captions_path = OUTPUT_DIR / "captions.md"
captions_text = ["# Didascalie delle figure", ""]
for title, caption in captions.items():
    captions_text.extend([f"## {title}", "", caption, ""])
captions_path.write_text("\n".join(captions_text), encoding="utf-8")

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "protocol": {
        "runs": int(len(runs)),
        "repeat_seeds": sorted(map(int, runs["repeat_seed"].unique())),
        "folds": sorted(map(int, runs["fold"].unique())),
        "inference_unit": "repeat_seed",
        "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
        "primary_tests": int(len(primary_hypotheses)),
        "secondary_tests": int(len(secondary_hypotheses)),
    },
    "inputs": {
        "full_archive": str(FULL_ARCHIVE),
        "full_archive_bytes": FULL_ARCHIVE.stat().st_size,
        "subset_archive": str(SUBSET_ARCHIVE),
        "subset_archive_bytes": SUBSET_ARCHIVE.stat().st_size,
    },
    "versions": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__,
        "seaborn": sns.__version__,
    },
    "figures": EXPORTED_FIGURES,
    "tables": [str(path) for path in EXPORTED_TABLES],
    "captions": str(captions_path),
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

zip_path = None
if MAKE_ZIP:
    archive_base = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_bundle"
    zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR))

expected_pairs = len(EXPORTED_FIGURES)
missing_exports = [
    name
    for item in EXPORTED_FIGURES
    for name in (item["png"], item["pdf"])
    if not Path(name).is_file() or Path(name).stat().st_size == 0
]
if missing_exports:
    raise RuntimeError(f"Esportazioni mancanti o vuote: {missing_exports}")

display(Markdown("\n".join([
    "## Analisi completata",
    "",
    f"- Figure: **{expected_pairs}** coppie PNG/PDF",
    f"- Tabelle CSV: **{len(EXPORTED_TABLES)}**",
    f"- Output: `{OUTPUT_DIR}`",
    f"- Archivio ZIP: `{zip_path}`" if zip_path else "- Archivio ZIP: disattivato",
])))
print({
    "output": str(OUTPUT_DIR),
    "figure_pairs": expected_pairs,
    "tables": len(EXPORTED_TABLES),
    "zip": str(zip_path) if zip_path else None,
    "validation": "ok",
})